# Practical 7: Cell-Cell Communication

Author: Francesca Drummer, Claudio Novella-Rausell

In this notebook we will revocer cell-cell communication (CCC) in spatial transcriptomics. We will test different ways doing so:
    1. Non-spatial CCC testing
    2. Spatially DE genes
    3. Spatially-informed CC

In [ ]:
import scanpy as sc
import squidpy as sq 
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.sparse import issparse, csr_matrix
import matplotlib.pyplot as plt
%matplotlib inline 
from liana.method import singlecellsignalr, connectome, cellphonedb, natmi, logfc, cellchat, geometric_mean

## 0. Download data

We will use the **Xenium AD dataset** from the previous notebooks here.

As a reminder the dataset consists of 6 coronal mouse brain slices from 2 different conditions (wildtype - ctrl vs TgCRND8 - AD) across 3 timepoints. In this practical, we additionally have information about cell types available in  `adata.obs['cell_types']`. Please note that these annotation are not perfect. For example, there are quite some cells that could not be assigned to a cell type (NaN or "unkown"). 

In this practical we aim to understand the differences of the mouse brain between the two conditions and across the timepoints using niches and spatial domains.

In [ ]:
PATH = "data"

In [ ]:
# load adata
adata = sc.read_h5ad(Path(PATH, 'xenium_mouse_ad_annotated_rotated_v2.h5ad'))
adata

In [ ]:
# load spatial domains from previous practical
adata.obs["spatial_domain_temp"] = pd.read_csv(Path(PATH, 'spatial_domains.csv'), index_col=0).astype(str)
adata = adata[~adata.obs["spatial_domain_temp"].isna()].copy() # let's filter out cells to match practical 6. These were probably filtered out prior to scVI

In [ ]:
print(adata.obs[['cell_id']].head())
print(adata.obs[['cell_labels']].value_counts())
print(adata.obs[['condition']].value_counts())
print(adata.obs[['time']].value_counts())
print(adata.obs[['batch_key']].value_counts())
print(adata.obs[['celltype']].value_counts())
print(adata.obs[['sample']].value_counts())

In [ ]:
adata.obs.groupby('time')['condition'].value_counts()

In [ ]:
print(adata.X[:10,:10])

In [ ]:
adata.layers['counts'] = adata.X

In [ ]:
# Normalization to the median
sc.pp.normalize_total(adata)

# Freeman-Tukey square root transform
assert issparse(adata.X)
sqrt_X = adata.X.sqrt()
# Create a new sparse matrix for X + 1
X_plus_1 = adata.X + csr_matrix(np.ones(adata.X.shape))
# Calculate the square root of (X + 1)
sqrt_X_plus_1 = X_plus_1.sqrt()
adata.layers['median_ft'] = sqrt_X + sqrt_X_plus_1

## 0. Introduction to LIANA+

[LIANA+](https://liana-py.readthedocs.io/en/latest/notebooks/basic_usage.html) is a toolbox in Python for various dissociated, multimodal and spatially informed cell-cell communication tools [Dimitrov et al., 2024]. 

First we install the package and observe which methods are implemented in LIANA+. 
Each method relies on different assumptions and returns a different ligand-receptor score. 
Usually, one score for the strength of the interaction (`magnitude`) and a score reflecting the `specifivity` of a interaction to a pair of cell identities. 

In [ ]:
import liana as li

li.mt.show_methods()

Most CCC tools identify LR interaction. For this they rely on a extracting LR pairs from databases. There are diverse databases but LIANA+ has a consensus database that uses LR that are overlapping across databases. 

First, we need to ensure that there are LR-pairs present in the data to be detected for communication.

In [ ]:
print(li.resource.show_resources())
resource_name = "mouseconsensus"  # Replace with the desired resource name if needed
lr_pairs = li.resource.select_resource(resource_name)
lr_pairs

In [ ]:
def lr_pairs_in_adata(adata):
    genes_in_dataset = set(adata.var_names)  # Replace `adata.var_names` with your dataset's gene names if different
    
    # Filter the ligand-receptor pairs for those present in your dataset
    filtered_lr_pairs = lr_pairs[
        lr_pairs['ligand'].isin(genes_in_dataset) & lr_pairs['receptor'].isin(genes_in_dataset)
    ]
    
    return filtered_lr_pairs
    

In [ ]:
filtered_lr_pairs = lr_pairs_in_adata(adata)

<span style="color:red; font-weight:bold">Task 1: How many ligand-receptor pairs are in the data?</span>

<details>
<summary>💡 Show solution</summary>
    
```python
# Count the number of ligand-receptor pairs present in your dataset
num_lr_pairs = len(filtered_lr_pairs)

# Display the count
print(f"Number of ligand-receptor pairs present in the dataset: {num_lr_pairs}")
```

In the following chapter, we will work with the CellPhoneDB method from LIANA+.

## 1. CellPhoneDB: non-spatial CCC

First, we will run CellPhoneDB as if we did not have any spatial information. 

In [ ]:
sub_adata = adata[(adata.obs['time'] == '5_7') & (adata.obs['condition'] == 'TgCRND8')]
sub_adata

In [ ]:
cellphonedb(sub_adata,
            groupby='celltype',
            # NOTE by default the resource uses HUMAN gene symbols
            resource_name='mouseconsensus',
            expr_prop=0.1,
            verbose=True, 
            use_raw = False,
            layer = 'counts',
            key_added='cpdb_res')

In [ ]:
print(sub_adata.uns)

In [ ]:
sub_adata.uns['cpdb_res'].head(10)
#sub_adata.uns['cpdb_res'].loc[(sub_adata.uns['cpdb_res']['ligand'] == 'C1qa') &
#                             (sub_adata.uns['cpdb_res']['receptor'] == 'Cd93')]

<div style="border: 1px solid #0000ff; padding: 10px; border-radius: 5px;">
<span style="color: #0000ff; font-size: 20px;"><b>Interpretation</b></span> <span style="font-size: 20px;">Liana+ scores</span>  

<span></span>
<ul>
    <li>source and target columns represent the source/sender and target/receiver cell identity for each interaction, respectively</li>
    <li>*_props: represents the proportion of cells that express the entity.</li>
    <li>*_means: entity expression mean per cell type.</li>
    <li>lr_means: mean ligand-receptor expression, as a measure of ligand-receptor interaction magnitude</li>
</ul>

<span style="color:red; font-weight:bold">Task 2: Plot the top 3 interacting complexes</span>

<details>
<summary>💡 Show solution</summary>
    
```python
sq.pl.spatial_scatter(sub_adata, 
                      color=[],
                      layer = 'median_ft',
                      shape=None)
```

In [ ]:
my_plot = li.pl.tileplot(adata = sub_adata,
                         fill='means',
                         label='props',
                         label_fun=lambda x: f'{x:.2f}',
                         top_n=30,
                         orderby='cellphone_pvals',
                         orderby_ascending=True,
                         source_labels=['Astrocytes', 'Oligodendrocytes', 'Microglia', 'Mixed Neurons/Oligo', 'Inhibitory neurons', 'Excitatory neurons'],
                         target_labels=['Astrocytes', 'Oligodendrocytes', 'Microglia', 'Mixed Neurons/Oligo', 'Inhibitory neurons', 'Excitatory neurons'],
                         uns_key='cpdb_res', # NOTE: default is 'liana_res'
                         source_title='Ligand',
                         target_title='Receptor',
                         figure_size=(8, 7)
                         )
my_plot

<span style="color:red; font-weight:bold">Question: What can we observe if we do not consider spatial information? Why could this be problematic?</span>

To overcome this issue we will cover two possible approaches to integrate spatial information into non-spatially aware CCC tools, like `CellPhoneDB`.

1. Restrict the input to spatially variable genes. 
2. Post-processing of interactions using spatial proximity, i.e. niche information. 

### Spatially-variable gene selection

We use Moran's I score as a measure of spatial autocorrelation to identify spatially variable genes. 

For more information see: [Chapter 29: Spatially variable genes](https://www.sc-best-practices.org/spatial/spatially_variable_genes.html) from single-cell best practices.

1. Calculate a spatial graph (`sq.gr.spatial_neighbors`)
2. Calculate autocorrelation with [Morans I score](https://squidpy.readthedocs.io/en/stable/notebooks/examples/graph/compute_moran.html) (`sq.gr.spatial_autocorr`)

In [ ]:
print(sub_adata.X[:5,:5])

In [ ]:
sq.gr.spatial_neighbors(sub_adata, n_neighs=30, coord_type="generic", key_added = 'neighs_based_spatial')

In [ ]:
sq.gr.interaction_matrix(sub_adata, cluster_key='celltype', connectivity_key = 'neighs_based_spatial', normalized=True)

In [ ]:
sq.pl.interaction_matrix(sub_adata, cluster_key='celltype')

In [ ]:
sq.gr.spatial_autocorr(sub_adata, connectivity_key = "neighs_based_spatial_connectivities", mode="moran", n_perms=50, genes=sub_adata.var_names)

Show and plot the top genes according to Moran's I score autocorrelation.

In [ ]:
sub_adata.uns["moranI"].head()

<div style="border: 1px solid #0000ff; padding: 10px; border-radius: 5px;">
<span style="color: #0000ff; font-size: 20px;"><b>Moran's I score</b></span> <span style="font-size: 20px;"></span>  

<span></span>
<ul>
    <li>I so the Moran’s I,</li>
    <li>pval_norm a p-value under normality assumption.</li>
    <li>var_norm the variance of the Moran’s I under normality assumption.</li>
    <li>{p_val}_{corr_method} the corrected p-values.</li>
</ul>

<span style="color:red; font-weight:bold">Task 3: Plot the 3 genes with the highest I score.</span>

<details>
<summary>💡 Show solution</summary>
    
```python
sq.pl.spatial_scatter(sub_adata, 
                      color=[],
                      layer = 'median_ft',
                      shape=None)
```

<span style="color:red; font-weight:bold">Task 4: Subset the data to include only genes that have a Morans I score higher than 0,2 and check that there are still relevant ligand-receptor pairs in the subdata.</span>

<details>
<summary>💡 Show solution</summary>
    
```python
sub_adata_svg = sub_adata[:, sub_adata.uns["moranI"]['I'] > 0.2]
sub_adata_svg
lr_pairs_in_adata(sub_adata_svg)
```

#### CellPhoneDB

In [ ]:
cellphonedb(sub_adata_svg,
            groupby='celltype',
            # NOTE by default the resource uses HUMAN gene symbols
            resource_name='mouseconsensus',
            expr_prop=0.1,
            verbose=True, 
            use_raw = False,
            layer = 'counts',
            key_added='cpdb_res')

In [ ]:
sub_adata_svg.uns['cpdb_res']

In [ ]:
my_plot = li.pl.tileplot(adata = sub_adata_svg,
                         fill='means',
                         label='props',
                         label_fun=lambda x: f'{x:.2f}',
                         top_n=20,
                         orderby='cellphone_pvals',
                         orderby_ascending=True,
                         source_labels=['Astrocytes', 'Oligodendrocytes', 'Microglia', 'Mixed Neurons/Oligo', 'Inhibitory neurons', 'Excitatory neurons'],
                         target_labels=['Astrocytes'],
                         uns_key='cpdb_res', # NOTE: default is 'liana_res'
                         source_title='Ligand',
                         target_title='Receptor',
                         figure_size=(8, 7)
                         )
my_plot

<span style="color:red; font-weight:bold">Question: What could be a potential limitation / problem with this approach?</span>

<span style="color:red; font-weight:bold">Optional Task: Compare the results for the healthy control or different time points. Do the CCC across cell types change?.</span>

<span style="color:red; font-weight:bold">Optional Task: Change the `expr_prop` in the CellPhoneDB function and try out some other tools like CellChat. How does it effect the results?.</span>

### Spatial proximity

An alternative to pre-selecting spatially variable genes is by restricting the cells to be spatially close when they are communicating. For this we will be using the calculated spatial domains from the previous tutorial. 

In [ ]:
sq.pl.spatial_scatter(sub_adata,
                      color = ['celltype', 'spatial_domain_temp'],
                      shape=None)

<span style="color:red; font-weight:bold">Task 5: Choose a spatial domain cluster that contains a high proportion of the cell types you are interested in to understand the interaction. Tip: also check that the fraction of unknown cells is low. </span>

In [ ]:
def relative_abundances(adata, group_by, cell_type_key):
    counts = adata.obs.groupby([group_by, cell_type_key]).size().unstack(fill_value=0)
    relative_abundance = counts.div(counts.sum(axis=1), axis=0)
    return relative_abundance

In [ ]:
relative_abundances(sub_adata, group_by='spatial_domain_temp', cell_type_key='celltype')

In [ ]:
## TODO
domain = 

In [ ]:
sub_adata_domain = sub_adata[sub_adata.obs['spatial_domain_temp'] == domain]
sub_adata_domain

In [ ]:
cellphonedb(sub_adata_domain,
            groupby='celltype',
            # NOTE by default the resource uses HUMAN gene symbols
            resource_name='mouseconsensus',
            expr_prop=0.1,
            verbose=True, 
            use_raw = False,
            layer = 'counts',
            key_added='cpdb_res')

In [ ]:
sub_adata_domain.uns['cpdb_res'].head()

In [ ]:
my_plot = li.pl.tileplot(adata = sub_adata_domain,
                         fill='means',
                         label='props',
                         label_fun=lambda x: f'{x:.2f}',
                         top_n=20,
                         orderby='cellphone_pvals',
                         orderby_ascending=True,
                         source_labels=['Astrocytes', 'Microglia',  'Oligodendrocytes'],
                         target_labels=['Astrocytes', 'Microglia', 'Oligodendrocytes'],
                         uns_key='cpdb_res', # NOTE: default is 'liana_res'
                         source_title='Ligand',
                         target_title='Receptor',
                         figure_size=(8, 7)
                         )
my_plot